In [1]:
import os

print("Installing required packages...")
!pip install -q flask pyngrok transformers torch torchaudio safetensors datasets accelerate jiwer torchcodec

print("\nMounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

#Create directories for the Flask app
print("Creating app directories...")
os.makedirs("templates", exist_ok=True)
os.makedirs("static", exist_ok=True)

Installing required packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 105.8 MB/s eta 0:00:00

Mounting Google Drive...
Mounted at /content/drive
Creating app directories...


In [2]:
# This file contains the model definitions and the main ensemble class
print("Creating model.py...")
model_py_content = """
import torch
import torchaudio
import os
import torch.nn as nn
import torch.nn.functional as F
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC, Wav2Vec2ConformerForCTC
from safetensors.torch import load_file
import warnings
from datasets import load_dataset, Audio
import jiwer


warnings.filterwarnings("ignore")

# Custom ASR Model (cnn-rnn-ctc)
class ASRModel(nn.Module):
    def __init__(self, n_mels, rnn_dim, vocab_size, n_cnn_layers, n_rnn_layers, dropout):
        super().__init__()
        cnn_channels = 32
        self.cnn = nn.Sequential(
            nn.Conv2d(1, cnn_channels, kernel_size=3, padding=1, stride=1), nn.ReLU(),
            nn.BatchNorm2d(cnn_channels),
            nn.Conv2d(cnn_channels, cnn_channels, kernel_size=3, padding=1, stride=1), nn.ReLU(),
            nn.BatchNorm2d(cnn_channels),
            nn.MaxPool2d(kernel_size=(2, 1)),
        )
        cnn_output_dim = (n_mels // 2) * cnn_channels
        self.rnn = nn.GRU(
            input_size=cnn_output_dim, hidden_size=rnn_dim,
            num_layers=n_rnn_layers, bidirectional=True,
            batch_first=True, dropout=dropout if n_rnn_layers > 1 else 0
        )
        self.classifier = nn.Linear(rnn_dim * 2, vocab_size)

    def forward(self, input_values, **kwargs):
        x = input_values.unsqueeze(1)
        x = self.cnn(x)
        batch_size, channels, freq_dim, seq_len = x.size()
        x = x.permute(0, 3, 1, 2).contiguous().view(batch_size, seq_len, -1)
        x, _ = self.rnn(x)
        logits = self.classifier(x)
        return {"logits": logits}

# Fusion Model (gru-rnn)
class FusionModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.GRU(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=2, bidirectional=True, batch_first=True, dropout=0.2
        )
        self.classifier = nn.Linear(hidden_size * 2, output_size)
        self.hidden_size = hidden_size

    def forward(self, x):
        self.rnn.flatten_parameters()
        x, _ = self.rnn(x)
        logits = self.classifier(x)
        return logits

# Main Ensemble Class (adapted for inference)
class InferenceEnsemble:
    def __init__(self, conformer_path, wav2vec2_path, custom_model_path, fusion_model_path):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Initializing system on device: {self.device}")

        print("Loading base models and processor...")
        self.processor = Wav2Vec2Processor.from_pretrained(wav2vec2_path)

        self.conformer_model = Wav2Vec2ConformerForCTC.from_pretrained(conformer_path).to(self.device).eval()
        self.wav2vec2_model = Wav2Vec2ForCTC.from_pretrained(wav2vec2_path).to(self.device).eval()

        self.custom_model = ASRModel(n_mels=80, rnn_dim=512, vocab_size=29, n_cnn_layers=3, n_rnn_layers=3, dropout=0.1)
        weights_path = os.path.join(custom_model_path, "model.safetensors")
        self.custom_model.load_state_dict(load_file(weights_path, device="cpu"))
        self.custom_model.to(self.device).eval()
        print("Base models loaded.")

        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=16000, n_fft=400, hop_length=160, n_mels=80
        ).to(self.device)

        fusion_model_input_size = 92
        self.fusion_model = FusionModel(
            input_size=fusion_model_input_size,
            hidden_size=512,
            output_size=self.processor.tokenizer.pad_token_id + 1
        ).to(self.device)

        self.fusion_model.load_state_dict(torch.load(fusion_model_path, map_location=self.device))
        self.fusion_model.eval()
        print(f"FusionModel initialized and loaded.")

    @torch.no_grad()
    def get_concatenated_logits(self, waveform):
        # ... (keep the existing code for this method) ...
        input_values = self.processor(waveform.cpu().numpy(), return_tensors="pt", sampling_rate=16000).input_values.to(self.device)

        logits_conformer = self.conformer_model(input_values).logits
        logits_wav2vec2 = self.wav2vec2_model(input_values).logits

        log_mel_spec = torch.log(torch.clamp(self.mel_transform(waveform.to(self.device)), min=1e-5))
        logits_custom = self.custom_model(log_mel_spec.unsqueeze(0))['logits']

        if not (logits_conformer.dim() == 3 and logits_wav2vec2.dim() == 3 and logits_custom.dim() == 3):
            raise ValueError("One of the models did not produce a 3D logits tensor.")

        min_len = min(logits_conformer.shape[1], logits_wav2vec2.shape[1], logits_custom.shape[1])
        logits_conformer = logits_conformer[:, :min_len, :]
        logits_wav2vec2 = logits_wav2vec2[:, :min_len, :]
        logits_custom = logits_custom[:, :min_len, :]

        transcription_conformer = self.processor.batch_decode(torch.argmax(logits_conformer, dim=-1))[0]
        transcription_wav2vec2 = self.processor.batch_decode(torch.argmax(logits_wav2vec2, dim=-1))[0]
        transcription_custom = self.processor.batch_decode(torch.argmax(logits_custom, dim=-1))[0]

        concatenated_logits = torch.cat([logits_conformer, logits_wav2vec2, logits_custom], dim=-1)

        return {
            "concatenated_logits": concatenated_logits,
            "transcription_conformer": transcription_conformer,
            "transcription_wav2vec2": transcription_wav2vec2,
            "transcription_custom": transcription_custom
        }

    def predict(self, audio_path):
        self.fusion_model.eval()
        waveform, sr = torchaudio.load(audio_path)
        if sr != 16000:
            waveform = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)(waveform)

        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        waveform = waveform.squeeze(0)

        with torch.no_grad():
            intermediate_results = self.get_concatenated_logits(waveform)
            final_logits = self.fusion_model(intermediate_results["concatenated_logits"])

        predicted_ids = torch.argmax(final_logits, dim=-1)
        final_transcription = self.processor.batch_decode(predicted_ids)[0]

        return {
            "Conformer": intermediate_results["transcription_conformer"],
            "Wav2Vec2": intermediate_results["transcription_wav2vec2"],
            "Custom Model": intermediate_results["transcription_custom"],
            "Final Ensemble": final_transcription
        }

    def evaluate(self, dataset_name, test_subset_size=2800):

        print(f"Loading dataset {dataset_name} for evaluation...")
        # --- MODIFIED LINE ---
        # The split name is updated to "Zulu_test" for the new dataset.
        full_test_dataset = load_dataset(dataset_name, split="Zulu_test")
        # --- END MODIFIED LINE ---
        test_dataset = full_test_dataset.select(range(min(test_subset_size, len(full_test_dataset))))

        test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16000))

        predictions = []
        references = []

        print("Starting evaluation...")
        for i, item in enumerate(test_dataset):
            print(f"Processing item {i+1}/{len(test_dataset)}...")
            audio_input = item["audio"]["array"]
            reference_text = item["transcription"]
            references.append(reference_text)

            waveform = torch.tensor(audio_input).unsqueeze(0)

            input_values = self.processor(waveform.cpu().numpy(), return_tensors="pt", sampling_rate=16000).input_values.to(self.device)
            logits_conformer = self.conformer_model(input_values).logits
            logits_wav2vec2 = self.wav2vec2_model(input_values).logits

            # This is the corrected part for the custom model
            mel_spec = self.mel_transform(waveform.to(self.device))
            log_mel_spec = torch.log(torch.clamp(mel_spec, min=1e-5))
            logits_custom = self.custom_model(log_mel_spec)['logits']

            # --- Pad/truncate logits to a common length ---
            min_len = min(logits_conformer.shape[1], logits_wav2vec2.shape[1], logits_custom.shape[1])
            logits_conformer = logits_conformer[:, :min_len, :]
            logits_wav2vec2 = logits_wav2vec2[:, :min_len, :]
            logits_custom = logits_custom[:, :min_len, :]

            # --- Concatenate and predict with the fusion model ---
            concatenated_logits = torch.cat([logits_conformer, logits_wav2vec2, logits_custom], dim=-1)
            final_logits = self.fusion_model(concatenated_logits)

            predicted_ids = torch.argmax(final_logits, dim=-1)
            final_transcription = self.processor.batch_decode(predicted_ids)[0]
            predictions.append(final_transcription)

        wer = jiwer.wer(references, predictions)
        cer = jiwer.cer(references, predictions)

        results = {"Final Ensemble": {"WER": f"{wer:.4f}", "CER": f"{cer:.4f}"}}
        print(f"Results for Final Ensemble: WER={wer:.4f}, CER={cer:.4f}")

        return results
"""
with open("model.py", "w") as f:
    f.write(model_py_content)

print("Creating templates/index.html...")
html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Multi Agent ASR Ensemble</title>
    <script src="https://cdn.tailwindcss.com"></script>
    <style>
        body { font-family: 'Inter', sans-serif; }
        @import url('https://rsms.me/inter/inter.css');
        .result-card, .eval-card {
            transition: all 0.3s ease-in-out;
        }
    </style>
</head>
<body class="bg-gray-100 flex items-center justify-center min-h-screen py-12">
    <div class="bg-white p-8 rounded-xl shadow-lg w-full max-w-2xl m-4">
        <h1 class="text-3xl font-bold text-gray-800 mb-2">Multi Agent ASR Ensemble</h1>
        <p class="text-gray-600 mb-6">Upload an audio file (.wav, .mp3) to get the transcription from the model.</p>

        <form id="upload-form" class="space-y-6">
            <div>
                <label for="audio-file" class="block text-sm font-medium text-gray-700">Audio File</label>
                <div class="mt-1 flex justify-center px-6 pt-5 pb-6 border-2 border-gray-300 border-dashed rounded-md">
                    <div class="space-y-1 text-center">
                        <svg class="mx-auto h-12 w-12 text-gray-400" stroke="currentColor" fill="none" viewBox="0 0 48 48" aria-hidden="true">
                            <path d="M28 8H12a4 4 0 00-4 4v20m32-12v8m0 0v8a4 4 0 01-4 4H12a4 4 0 01-4-4v-4m32-4l-3.172-3.172a4 4 0 00-5.656 0L28 28M8 32l9.172-9.172a4 4 0 015.656 0L28 28m0 0l4 4m4-24h8m-4-4v8" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" />
                        </svg>
                        <div class="flex text-sm text-gray-600">
                            <label for="audio-file-input" class="relative cursor-pointer bg-white rounded-md font-medium text-indigo-600 hover:text-indigo-500 focus-within:outline-none focus-within:ring-2 focus-within:ring-offset-2 focus-within:ring-indigo-500">
                                <span>Upload a file</span>
                                <input id="audio-file-input" name="audio" type="file" class="sr-only">
                            </label>
                            <p class="pl-1">or drag and drop</p>
                        </div>
                        <p id="file-name" class="text-xs text-gray-500">WAV, MP3 up to 10MB</p>
                    </div>
                </div>
            </div>

            <div>
                <button type="submit" id="submit-btn" class="w-full flex justify-center py-3 px-4 border border-transparent rounded-md shadow-sm text-sm font-medium text-white bg-indigo-600 hover:bg-indigo-700 focus:outline-none focus:ring-2 focus:ring-offset-2 focus:ring-indigo-500">
                    Transcribe
                </button>
            </div>
            <div>
                <button type="button" id="evaluate-btn" class="w-full flex justify-center py-3 px-4 border border-transparent rounded-md shadow-sm text-sm font-medium text-white bg-emerald-600 hover:bg-emerald-700 focus:outline-none focus:ring-2 focus-within:ring-offset-2 focus:ring-emerald-500">
                    Evaluate Ensemble Model on Test Set
                </button>
            </div>
        </form>

        <div id="results" class="mt-8 hidden">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Transcription Results</h2>
            <div id="loader" class="text-center py-4 hidden">
                <div class="animate-spin rounded-full h-8 w-8 border-b-2 border-indigo-500 mx-auto"></div>
                <p class="mt-2 text-gray-600">Processing audio, please wait...</p>
            </div>
            <div id="result-content" class="space-y-4"></div>
        </div>

        <div id="evaluation-results" class="mt-8 hidden">
            <h2 class="text-2xl font-semibold text-gray-800 mb-4">Evaluation Metrics (WER / CER)</h2>
            <div id="evaluation-loader" class="text-center py-4 hidden">
                <div class="animate-spin rounded-full h-8 w-8 border-b-2 border-emerald-500 mx-auto"></div>
                <p class="mt-2 text-gray-600">Evaluating on a test subset, this may take a few minutes...</p>
            </div>
            <div id="evaluation-content" class="space-y-4"></div>
        </div>
    </div>

    <script>
        const form = document.getElementById('upload-form');
        const fileInput = document.getElementById('audio-file-input');
        const fileNameDisplay = document.getElementById('file-name');
        const submitBtn = document.getElementById('submit-btn');

        const resultsDiv = document.getElementById('results');
        const loader = document.getElementById('loader');
        const resultContent = document.getElementById('result-content');

        const evaluateBtn = document.getElementById('evaluate-btn');
        const evalResultsDiv = document.getElementById('evaluation-results');
        const evalLoader = document.getElementById('evaluation-loader');
        const evalContent = document.getElementById('evaluation-content');

        fileInput.addEventListener('change', () => {
            if (fileInput.files.length > 0) {
                fileNameDisplay.textContent = fileInput.files[0].name;
            } else {
                fileNameDisplay.textContent = 'WAV, MP3 up to 10MB';
            }
        });

        form.addEventListener('submit', async (e) => {
            e.preventDefault();
            if (fileInput.files.length === 0) {
                alert('Please select an audio file first.');
                return;
            }

            const formData = new FormData();
            formData.append('audio', fileInput.files[0]);

            resultsDiv.classList.remove('hidden');
            evalResultsDiv.classList.add('hidden'); // Hide eval results
            loader.classList.remove('hidden');
            resultContent.innerHTML = '';
            submitBtn.disabled = true;
            evaluateBtn.disabled = true;
            submitBtn.classList.add('opacity-50', 'cursor-not-allowed');
            evaluateBtn.classList.add('opacity-50', 'cursor-not-allowed');

            try {
                const response = await fetch('/predict', {
                    method: 'POST',
                    body: formData,
                });

                if (!response.ok) {
                    const error = await response.json();
                    throw new Error(error.error || 'Something went wrong');
                }

                const data = await response.json();

                let html = '';
                const finalTranscription = data['Final Ensemble'];

                if (finalTranscription) {
                    html += `
                    <div class="p-4 rounded-lg bg-indigo-50 border-l-4 border-indigo-500">
                        <h3 class="text-lg font-semibold text-indigo-800">Final Ensemble</h3>
                        <p class="mt-1 text-md text-indigo-900">${finalTranscription}</p>
                    </div>
                    `;
                } else {
                     html += `
                    <div class="p-4 rounded-lg bg-red-50 border-l-4 border-red-500">
                        <h3 class="text-lg font-semibold text-red-800">Error</h3>
                        <p class="mt-1 text-md text-red-900">Final Ensemble transcription not found in response.</p>
                    </div>
                    `;
                }
                resultContent.innerHTML = html;

            } catch (error) {
                resultContent.innerHTML = `<div class="p-4 rounded-lg bg-red-50 border-l-4 border-red-500">
                    <h3 class="text-lg font-semibold text-red-800">Error</h3>
                    <p class="mt-1 text-md text-red-900">${error.message}</p>
                </div>`;
            } finally {
                loader.classList.add('hidden');
                submitBtn.disabled = false;
                evaluateBtn.disabled = false;
                submitBtn.classList.remove('opacity-50', 'cursor-not-allowed');
                evaluateBtn.classList.remove('opacity-50', 'cursor-not-allowed');
            }
        });

        evaluateBtn.addEventListener('click', async () => {
            evalResultsDiv.classList.remove('hidden');
            resultsDiv.classList.add('hidden'); // Hide transcription results
            evalLoader.classList.remove('hidden');
            evalContent.innerHTML = '';
            evaluateBtn.disabled = true;
            submitBtn.disabled = true;
            evaluateBtn.classList.add('opacity-50', 'cursor-not-allowed');
            submitBtn.classList.add('opacity-50', 'cursor-not-allowed');

            try {
                const response = await fetch('/evaluate', { method: 'POST' });

                if (!response.ok) {
                    const error = await response.json();
                    throw new Error(error.error || 'An unknown error occurred during evaluation');
                }

                const data = await response.json();
                let html = '<div class="grid grid-cols-1 gap-4">';

                for (const modelName in data) {
                    html += `<div class="eval-card p-4 rounded-lg bg-emerald-50 border-t-4 border-emerald-500 text-center shadow-md">
                        <h3 class="text-lg font-bold text-emerald-800">${modelName}</h3>
                        <div class="mt-2"><p class="text-sm text-gray-600">Word Error Rate</p><p class="text-2xl font-semibold text-gray-900">${data[modelName].WER}</p></div>
                        <div class="mt-2"><p class="text-sm text-gray-600">Character Error Rate</p><p class="text-2xl font-semibold text-gray-900">${data[modelName].CER}</p></div>
                    </div>`;
                }
                html += '</div>';
                evalContent.innerHTML = html;

            } catch (error) {
                evalContent.innerHTML = `<div class="p-4 rounded-lg bg-red-50 border-l-4 border-red-500"><h3 class="text-lg font-semibold text-red-800">Evaluation Error</h3><p class="mt-1 text-md text-red-900">${error.message}</p></div>`;
            } finally {
                evalLoader.classList.add('hidden');
                evaluateBtn.disabled = false;
                submitBtn.disabled = false;
                evaluateBtn.classList.remove('opacity-50', 'cursor-not-allowed');
                submitBtn.classList.remove('opacity-50', 'cursor-not-allowed');
            }
        });
    </script>
</body>
</html>
"""
with open("templates/index.html", "w") as f:
    f.write(html_content)

# --- 6. Write the Flask App (app.py) ---
print("Creating app.py...")
app_py_content = """
from flask import Flask, request, render_template, jsonify
from pyngrok import ngrok
import os
from model import InferenceEnsemble
import torch

NGROK_AUTH_TOKEN = "32NqxOiMohuBmUDNcFK3fAHyzvK_4Vwcpnq7PUQWR4qF3c9Xc"

if not NGROK_AUTH_TOKEN:
    print("ERROR: Ngrok authtoken not set.")
    print("Please get your token from https://dashboard.ngrok.com/get-started/your-authtoken and paste it into the NGROK_AUTH_TOKEN variable.")
else:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

app = Flask(__name__)

CONFORMER_MODEL_PATH = "/content/drive/MyDrive/wav2vecconformer-finetune-checkpoints"
WAV2VEC2_MODEL_PATH = "/content/drive/MyDrive/wav2vec2-finetune-checkpoints22/checkpoint-20000"
CUSTOM_MODEL_PATH = "/content/drive/MyDrive/my_final_wav2vec2_model"
FUSION_MODEL_PATH = "/content/drive/MyDrive/trained_fusion_model.pth"

print("Starting model loading process...")
try:
    ensemble_system = InferenceEnsemble(
        conformer_path=CONFORMER_MODEL_PATH,
        wav2vec2_path=WAV2VEC2_MODEL_PATH,
        custom_model_path=CUSTOM_MODEL_PATH,
        fusion_model_path=FUSION_MODEL_PATH
    )
    print("\\nModel loaded successfully! The app is ready.")
except Exception as e:
    print(f"\\nError loading model: {e}")
    print("Please ensure the model paths are correct and the files are accessible.")
    ensemble_system = None

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/predict', methods=['POST'])
def predict():
    if ensemble_system is None:
        return jsonify({'error': 'Model is not loaded. Please check the server logs.'}), 500

    if 'audio' not in request.files:
        return jsonify({'error': 'No audio file found'}), 400

    file = request.files['audio']
    if file.filename == '':
        return jsonify({'error': 'No selected file'}), 400

    temp_path = "temp_audio_file.wav"
    try:
        file.save(temp_path)
        result = ensemble_system.predict(temp_path)
        os.remove(temp_path)
        return jsonify(result)
    except Exception as e:
        if os.path.exists("temp_audio_file.wav"):
            os.remove("temp_audio_file.wav")
        print(f"Error during prediction: {e}")
        return jsonify({'error': str(e)}), 500

@app.route('/evaluate', methods=['POST'])
def evaluate():
    if ensemble_system is None:
        return jsonify({'error': 'Model is not loaded. Check server logs for errors.'}), 500
    try:
        # --- MODIFIED LINE ---
        # The dataset name is changed to Bejuka/Zulu_test.
        dataset_name = "Beijuka/Zulu_test"
        # --- END MODIFIED LINE ---
        results = ensemble_system.evaluate(dataset_name)
        return jsonify(results)
    except Exception as e:
        print(f"Error during evaluation: {e}")
        return jsonify({'error': str(e)}), 500

if __name__ == '__main__':
    if NGROK_AUTH_TOKEN:
        public_url = ngrok.connect(5000)
        print(f" * ngrok tunnel is active at: {public_url}")
        app.run()
    else:
        print("Flask app did not start because NGROK_AUTH_TOKEN is not set.")
"""
with open("app.py", "w") as f:
    f.write(app_py_content)

print("\n--- Setup Complete ---")

Creating model.py...
Creating templates/index.html...
Creating app.py...

--- Setup Complete ---


In [3]:
!python app.py

2025-09-17 10:09:09.506433: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-17 10:09:09.523686: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758103749.545079    1126 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758103749.551520    1126 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758103749.568386    1126 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 